# USE CASE 4: Risk-Informed Development Planning

## For: Ministry of Planning, Urban Development, Infrastructure Ministries

### Key Questions This Analysis Answers:
1. **Which areas should be restricted from development due to high hazard exposure?**
2. **Where should we prioritize infrastructure investments for resilience?**
3. **How can we screen development policies for disaster risk?**
4. **Which locations need climate adaptation interventions?**
5. **What land use zoning should be based on multi-hazard exposure?**

### Why Historical Data Matters:
- **Evidence-Based Zoning** → Restrict development in high-risk areas
- **Infrastructure Planning** → Build resilient infrastructure where needed
- **Policy Screening** → Ensure development doesn't increase risk
- **Adaptation Targeting** → Focus climate adaptation where most effective

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from config import PROCESSED_DATA_DIR, FIGURES_DIR, REPORTS_DIR

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Libraries loaded")

✓ Libraries loaded


## 1. Load Risk Assessment Data

In [2]:
data_file = PROCESSED_DATA_DIR / 'disaster_events_sendai.csv'
risk_file = REPORTS_DIR / 'multi_hazard_risk_assessment.csv'

if data_file.exists():
    df = pd.read_csv(data_file)
    print(f"✓ Loaded {len(df)} disaster events")
else:
    print("Please run notebook 05_risk_modeling.ipynb first")
    df = None

if risk_file.exists():
    risk_df = pd.read_csv(risk_file, index_col=0)
    print(f"✓ Loaded risk assessment for {len(risk_df)} locations")
else:
    print("Risk assessment file not found")
    risk_df = None

✓ Loaded 30 disaster events
✓ Loaded risk assessment for 25 locations


## 2. Land Use Zoning Based on Multi-Hazard Exposure

### Planning Message:
"Historical disaster patterns define which areas are suitable for development."

In [3]:
if risk_df is not None and 'risk_score' in risk_df.columns:
    # Define land use zones based on risk scores
    risk_df['land_use_zone'] = pd.cut(
        risk_df['risk_score'],
        bins=[0, 0.3, 0.5, 0.7, 1.0],
        labels=['LOW RISK - Unrestricted Development',
                'MODERATE RISK - Conditional Development',
                'HIGH RISK - Restricted Development',
                'VERY HIGH RISK - No New Development']
    )
    
    print("="*80)
    print("LAND USE ZONING RECOMMENDATIONS BASED ON MULTI-HAZARD EXPOSURE")
    print("="*80)
    
    zone_counts = risk_df['land_use_zone'].value_counts()
    
    print("\nLand use zone distribution:\n")
    for zone, count in zone_counts.items():
        pct = (count / len(risk_df)) * 100
        print(f"{zone}: {count} locations ({pct:.1f}%)")
    
    # Detailed recommendations by zone
    print("\n" + "="*80)
    print("PLANNING GUIDELINES BY RISK ZONE")
    print("="*80)
    
    print("\n🟢 LOW RISK ZONES (Risk Score < 0.3):")
    low_risk = risk_df[risk_df['risk_score'] < 0.3].sort_values('risk_score').head(10)
    if len(low_risk) > 0:
        print("   Suitable for:")
        print("   - Residential development")
        print("   - Critical infrastructure (hospitals, schools)")
        print("   - Industrial zones")
        print("   - Commercial centers")
        print(f"\n   Top locations: {', '.join(low_risk.head(5).index)}")
    
    print("\n🟡 MODERATE RISK ZONES (Risk Score 0.3-0.5):")
    print("   Development conditions:")
    print("   - Require disaster risk assessment for new projects")
    print("   - Mandatory building codes for resilience")
    print("   - Infrastructure must include DRR measures")
    print("   - Restrict critical facilities (hospitals, emergency centers)")
    
    print("\n🟠 HIGH RISK ZONES (Risk Score 0.5-0.7):")
    print("   Development restrictions:")
    print("   - NO critical infrastructure (hospitals, schools, emergency centers)")
    print("   - Residential development only with enhanced building standards")
    print("   - Mandatory evacuation plans for all developments")
    print("   - Require DRR investment equal to 10-20% of project cost")
    
    print("\n🔴 VERY HIGH RISK ZONES (Risk Score > 0.7):")
    very_high_risk = risk_df[risk_df['risk_score'] > 0.7].sort_values('risk_score', ascending=False).head(10)
    if len(very_high_risk) > 0:
        print("   NO NEW DEVELOPMENT PERMITTED")
        print("   Recommended actions:")
        print("   - Relocate existing vulnerable populations")
        print("   - Convert to green spaces / buffer zones")
        print("   - Implement nature-based solutions")
        print("   - Restrict to agriculture (non-permanent structures only)")
        print(f"\n   Critical zones: {', '.join(very_high_risk.head(5).index)}")
    
    # Save zoning recommendations
    risk_df.to_csv(REPORTS_DIR / 'UC4_land_use_zoning.csv')
    print("\n✓ Land use zoning recommendations saved")

LAND USE ZONING RECOMMENDATIONS BASED ON MULTI-HAZARD EXPOSURE

Land use zone distribution:

LOW RISK - Unrestricted Development: 22 locations (88.0%)
HIGH RISK - Restricted Development: 2 locations (8.0%)
MODERATE RISK - Conditional Development: 0 locations (0.0%)
VERY HIGH RISK - No New Development: 0 locations (0.0%)

PLANNING GUIDELINES BY RISK ZONE

🟢 LOW RISK ZONES (Risk Score < 0.3):
   Suitable for:
   - Residential development
   - Critical infrastructure (hospitals, schools)
   - Industrial zones
   - Commercial centers

   Top locations: Kilimanjaro, Mwanza, Afar, Southern Nations, Coast

🟡 MODERATE RISK ZONES (Risk Score 0.3-0.5):
   Development conditions:
   - Require disaster risk assessment for new projects
   - Mandatory building codes for resilience
   - Infrastructure must include DRR measures
   - Restrict critical facilities (hospitals, emergency centers)

🟠 HIGH RISK ZONES (Risk Score 0.5-0.7):
   Development restrictions:
   - NO critical infrastructure (hospital

## 3. Infrastructure Investment Prioritization

In [4]:
if df is not None and 'level2' in df.columns:
    # Analyze infrastructure damage patterns
    infra_damage = df.groupby('level2').agg({
        'housesdestroyed': 'sum',
        'housesdamaged': 'sum',
        'educationcenters': 'sum',
        'healthcenters': 'sum',
        'serial': 'count'
    }).rename(columns={'serial': 'event_count'})
    
    infra_damage['total_infrastructure_damage'] = (
        infra_damage['housesdestroyed'] + 
        infra_damage['housesdamaged'] + 
        infra_damage['educationcenters'] + 
        infra_damage['healthcenters']
    )
    
    infra_damage['avg_damage_per_event'] = (
        infra_damage['total_infrastructure_damage'] / infra_damage['event_count']
    )
    
    # Prioritize based on damage frequency and magnitude
    infra_damage = infra_damage.sort_values('total_infrastructure_damage', ascending=False)
    
    print("\n" + "="*80)
    print("INFRASTRUCTURE INVESTMENT PRIORITIES")
    print("="*80)
    print("\nTop 20 locations requiring resilient infrastructure investment:\n")
    
    display(infra_damage.head(20))
    
    print("\n🏗️ INFRASTRUCTURE INVESTMENT RECOMMENDATIONS:\n")
    
    print("TIER 1 - CRITICAL INFRASTRUCTURE UPGRADES (Top 5):")
    for i, location in enumerate(infra_damage.head(5).index, 1):
        total_damage = infra_damage.loc[location, 'total_infrastructure_damage']
        health = infra_damage.loc[location, 'healthcenters']
        education = infra_damage.loc[location, 'educationcenters']
        
        print(f"\n{i}. {location}:")
        print(f"   Total infrastructure damaged: {total_damage:,.0f} units")
        print(f"   Health facilities: {health:,.0f} | Education: {education:,.0f}")
        print("   Recommended investments:")
        print("   → Retrofit existing critical facilities (hospitals, schools)")
        print("   → Build to enhanced resilience standards")
        print("   → Implement early warning systems")
        print("   → Develop evacuation infrastructure")
    
    print("\nTIER 2 - HOUSING RESILIENCE (Next 10):")
    for location in infra_damage.iloc[5:15].index:
        houses = infra_damage.loc[location, 'housesdestroyed'] + infra_damage.loc[location, 'housesdamaged']
        print(f"   {location}: {houses:,.0f} houses damaged")
    print("\n   → Implement building code enforcement")
    print("   → Provide resilient housing subsidies")
    print("   → Upgrade drainage and flood protection")
    
    infra_damage.to_csv(REPORTS_DIR / 'UC4_infrastructure_priorities.csv')
    print("\n✓ Infrastructure investment priorities saved")


INFRASTRUCTURE INVESTMENT PRIORITIES

Top 20 locations requiring resilient infrastructure investment:



,housesdestroyed,housesdamaged,educationcenters,healthcenters,event_count,total_infrastructure_damage,avg_damage_per_event
level2,,,,,,,
Bay,45000,123000,234,67,1,168301,168301.0
Western,12450,57200,72,19,2,69741,34870.5
Somali,12500,45000,89,23,1,57612,57612.0
Lower Shabelle,12000,45000,78,23,1,57101,57101.0
Rift Valley,9570,36300,53,14,2,45937,22968.5
Nairobi,8990,34200,65,17,3,43272,14424.0
Northern,8900,34000,56,15,1,42971,42971.0
North Eastern,8900,34000,12,3,1,42915,42915.0
Morogoro,5600,18000,34,9,1,23643,23643.0



🏗️ INFRASTRUCTURE INVESTMENT RECOMMENDATIONS:

TIER 1 - CRITICAL INFRASTRUCTURE UPGRADES (Top 5):

1. Bay:
   Total infrastructure damaged: 168,301 units
   Health facilities: 67 | Education: 234
   Recommended investments:
   → Retrofit existing critical facilities (hospitals, schools)
   → Build to enhanced resilience standards
   → Implement early warning systems
   → Develop evacuation infrastructure

2. Western:
   Total infrastructure damaged: 69,741 units
   Health facilities: 19 | Education: 72
   Recommended investments:
   → Retrofit existing critical facilities (hospitals, schools)
   → Build to enhanced resilience standards
   → Implement early warning systems
   → Develop evacuation infrastructure

3. Somali:
   Total infrastructure damaged: 57,612 units
   Health facilities: 23 | Education: 89
   Recommended investments:
   → Retrofit existing critical facilities (hospitals, schools)
   → Build to enhanced resilience standards
   → Implement early warning systems
   → De

## 4. Development Policy Risk Screening

### Risk Screening Matrix for New Development Projects

In [5]:
if df is not None and 'hazardtype' in df.columns and 'level2' in df.columns:
    # Create risk screening matrix
    print("\n" + "="*80)
    print("DEVELOPMENT POLICY RISK SCREENING FRAMEWORK")
    print("="*80)
    
    print("\nUse this framework to screen all new development proposals:\n")
    
    print("STEP 1: LOCATION RISK ASSESSMENT")
    print("   Question: Is the proposed location in a high-risk zone?")
    if risk_df is not None:
        high_risk_locations = risk_df[risk_df['risk_score'] > 0.5].index.tolist()
        print(f"   High-risk locations ({len(high_risk_locations)}): {', '.join(high_risk_locations[:10])}...")
    print("   → If YES: Require detailed risk assessment")
    print("   → If NO: Proceed to Step 2")
    
    print("\nSTEP 2: HAZARD EXPOSURE CHECK")
    print("   Question: Which hazards affect this location?")
    location_hazards = df.groupby('level2')['hazardtype'].apply(lambda x: ', '.join(x.unique())).head(5)
    print("   Example hazard profiles:")
    for loc, hazards in location_hazards.items():
        print(f"   - {loc}: {hazards}")
    print("   → Design must address ALL identified hazards")
    
    print("\nSTEP 3: INFRASTRUCTURE TYPE SCREENING")
    print("   Critical Infrastructure (hospitals, schools, emergency centers):")
    print("   → PROHIBITED in Very High Risk zones (risk score > 0.7)")
    print("   → RESTRICTED in High Risk zones (risk score 0.5-0.7)")
    print("   → CONDITIONAL in Moderate Risk zones (risk score 0.3-0.5)")
    print("   → PERMITTED in Low Risk zones (risk score < 0.3)")
    
    print("\nSTEP 4: RESILIENCE REQUIREMENTS")
    print("   Based on risk score, require:")
    print("   - Risk Score 0.3-0.5: Standard building codes + basic DRR")
    print("   - Risk Score 0.5-0.7: Enhanced building codes + 10% DRR investment")
    print("   - Risk Score > 0.7: Special approval + 20% DRR investment")
    
    print("\nSTEP 5: ENVIRONMENTAL IMPACT")
    print("   Question: Will development increase risk for others?")
    print("   → Assess downstream flood risk")
    print("   → Evaluate deforestation/landslide risk")
    print("   → Check impact on natural buffers")
    print("   → If YES to any: Require mitigation measures")
    
    # Create screening checklist
    screening_checklist = pd.DataFrame({
        'Screening Step': [
            '1. Location Risk Assessment',
            '2. Hazard Exposure Check',
            '3. Infrastructure Type Screening',
            '4. Resilience Requirements',
            '5. Environmental Impact Assessment'
        ],
        'Required Action': [
            'Check location against risk zones',
            'Identify all hazards affecting location',
            'Verify infrastructure type is permitted',
            'Apply appropriate building standards',
            'Assess and mitigate downstream risks'
        ],
        'Decision Point': [
            'Proceed / Require detailed assessment / Reject',
            'Design for identified hazards',
            'Permit / Restrict / Prohibit',
            'Standard / Enhanced / Special approval',
            'Approve / Require mitigation / Reject'
        ]
    })
    
    screening_checklist.to_csv(REPORTS_DIR / 'UC4_policy_screening_checklist.csv', index=False)
    print("\n✓ Policy screening checklist saved")


DEVELOPMENT POLICY RISK SCREENING FRAMEWORK

Use this framework to screen all new development proposals:

STEP 1: LOCATION RISK ASSESSMENT
   Question: Is the proposed location in a high-risk zone?
   High-risk locations (2): Bay, Lower Shabelle...
   → If YES: Require detailed risk assessment
   → If NO: Proceed to Step 2

STEP 2: HAZARD EXPOSURE CHECK
   Question: Which hazards affect this location?
   Example hazard profiles:
   - Addis Ababa: FLOOD
   - Afar: EARTHQUAKE
   - Banadir: DROUGHT
   - Bay: DROUGHT
   - Central: FLOOD
   → Design must address ALL identified hazards

STEP 3: INFRASTRUCTURE TYPE SCREENING
   Critical Infrastructure (hospitals, schools, emergency centers):
   → PROHIBITED in Very High Risk zones (risk score > 0.7)
   → RESTRICTED in High Risk zones (risk score 0.5-0.7)
   → CONDITIONAL in Moderate Risk zones (risk score 0.3-0.5)
   → PERMITTED in Low Risk zones (risk score < 0.3)

STEP 4: RESILIENCE REQUIREMENTS
   Based on risk score, require:
   - Risk S

## 5. Climate Adaptation Strategy Targeting

In [6]:
if df is not None and 'hazardtype' in df.columns:
    # Identify climate-related hazards
    climate_hazards = ['FLOOD', 'DROUGHT', 'STORM', 'HEAVY RAIN', 'EXTREME TEMPERATURE']
    
    climate_df = df[df['hazardtype'].isin(climate_hazards)]
    
    if len(climate_df) > 0:
        # Analyze climate adaptation needs by location
        adaptation_needs = climate_df.groupby('level2').agg({
            'serial': 'count',
            'hazardtype': lambda x: ', '.join(x.unique()),
            'total_affected': 'sum',
            'usdvalue': 'sum'
        }).rename(columns={
            'serial': 'climate_events',
            'hazardtype': 'climate_hazards',
            'total_affected': 'people_affected',
            'usdvalue': 'economic_loss'
        })
        
        adaptation_needs = adaptation_needs.sort_values('climate_events', ascending=False)
        
        print("\n" + "="*80)
        print("CLIMATE ADAPTATION STRATEGY TARGETING")
        print("="*80)
        print(f"\nAnalyzed {len(climate_df)} climate-related disaster events")
        print(f"Climate hazards: {', '.join(climate_hazards)}\n")
        
        print("Top 15 locations requiring climate adaptation interventions:\n")
        display(adaptation_needs.head(15))
        
        print("\n🌍 CLIMATE ADAPTATION RECOMMENDATIONS:\n")
        
        # Hazard-specific adaptation strategies
        hazard_strategies = {
            'FLOOD': [
                'Improve drainage systems',
                'Construct flood barriers/levees',
                'Restore wetlands and natural buffers',
                'Implement early warning systems'
            ],
            'DROUGHT': [
                'Develop water storage infrastructure',
                'Promote water-efficient agriculture',
                'Diversify water sources',
                'Implement drought-resistant crops'
            ],
            'STORM': [
                'Strengthen building codes',
                'Plant windbreaks and coastal vegetation',
                'Improve coastal zone management',
                'Develop cyclone shelters'
            ]
        }
        
        for hazard, strategies in hazard_strategies.items():
            hazard_count = climate_df[climate_df['hazardtype'] == hazard].shape[0]
            if hazard_count > 0:
                print(f"{hazard} ({hazard_count} events):")
                for strategy in strategies:
                    print(f"   → {strategy}")
                print()
        
        adaptation_needs.to_csv(REPORTS_DIR / 'UC4_climate_adaptation_targets.csv')
        print("✓ Climate adaptation targeting saved")


CLIMATE ADAPTATION STRATEGY TARGETING

Analyzed 24 climate-related disaster events
Climate hazards: FLOOD, DROUGHT, STORM, HEAVY RAIN, EXTREME TEMPERATURE

Top 15 locations requiring climate adaptation interventions:



,climate_events,climate_hazards,people_affected,economic_loss
level2,,,,
Nairobi,2,FLOOD,594575,14040000
Eastern,2,"FLOOD, DROUGHT",1191567,6840000
Bay,1,DROUGHT,1553900,106800000
Central,1,FLOOD,201234,2760000
Addis Ababa,1,FLOOD,132467,1068000
Banadir,1,DROUGHT,1200000,4200000
Dar es Salaam,1,STORM,97156,1800000
Coast,1,HEAVY RAIN,82123,1068000
Lower Shabelle,1,DROUGHT,1136000,14400000



🌍 CLIMATE ADAPTATION RECOMMENDATIONS:

FLOOD (11 events):
   → Improve drainage systems
   → Construct flood barriers/levees
   → Restore wetlands and natural buffers
   → Implement early warning systems

DROUGHT (7 events):
   → Develop water storage infrastructure
   → Promote water-efficient agriculture
   → Diversify water sources
   → Implement drought-resistant crops

STORM (5 events):
   → Strengthen building codes
   → Plant windbreaks and coastal vegetation
   → Improve coastal zone management
   → Develop cyclone shelters

✓ Climate adaptation targeting saved


## 6. Multi-Hazard Exposure Mapping for Planning

In [7]:
if df is not None and 'level2' in df.columns:
    # Create multi-hazard exposure profile
    hazard_matrix = df.groupby(['level2', 'hazardtype']).size().unstack(fill_value=0)
    
    # Calculate exposure diversity
    hazard_matrix['total_events'] = hazard_matrix.sum(axis=1)
    hazard_matrix['hazard_diversity'] = (hazard_matrix > 0).sum(axis=1)
    
    # Identify multi-hazard hotspots
    multi_hazard = hazard_matrix[hazard_matrix['hazard_diversity'] >= 3].sort_values('hazard_diversity', ascending=False)
    
    print("\n" + "="*80)
    print("MULTI-HAZARD EXPOSURE MAPPING")
    print("="*80)
    print(f"\nLocations exposed to 3+ hazard types: {len(multi_hazard)}\n")
    
    print("Top 15 multi-hazard hotspots:\n")
    display(multi_hazard.head(15))
    
    print("\n📍 PLANNING IMPLICATIONS FOR MULTI-HAZARD AREAS:\n")
    
    for location in multi_hazard.head(5).index:
        diversity = multi_hazard.loc[location, 'hazard_diversity']
        events = multi_hazard.loc[location, 'total_events']
        
        # Get specific hazards
        location_hazards = hazard_matrix.loc[location]
        active_hazards = location_hazards[location_hazards > 0].drop(['total_events', 'hazard_diversity']).index.tolist()
        
        print(f"{location}:")
        print(f"   Exposed to {diversity} hazard types ({events} total events)")
        print(f"   Hazards: {', '.join(active_hazards[:5])}")
        print("   Planning requirements:")
        print("   → INTEGRATED multi-hazard risk assessment mandatory")
        print("   → Development must address ALL hazards")
        print("   → Consider relocation for critical infrastructure")
        print("   → Implement comprehensive DRR measures\n")
    
    multi_hazard.to_csv(REPORTS_DIR / 'UC4_multi_hazard_exposure.csv')
    print("✓ Multi-hazard exposure mapping saved")


MULTI-HAZARD EXPOSURE MAPPING

Locations exposed to 3+ hazard types: 4

Top 15 multi-hazard hotspots:



hazardtype,DROUGHT,EARTHQUAKE,EPIDEMIC,FIRE,FLOOD,HEAVY RAIN,LANDSLIDE,STORM,total_events,hazard_diversity
level2,,,,,,,,,,
Eastern,1,0,0,0,1,0,0,0,2,3
Nairobi,0,0,0,1,2,0,0,0,3,3
Rift Valley,0,0,1,0,1,0,0,0,2,3
Western,0,0,0,0,1,0,1,0,2,3



📍 PLANNING IMPLICATIONS FOR MULTI-HAZARD AREAS:

Eastern:
   Exposed to 3 hazard types (2 total events)
   Hazards: DROUGHT, FLOOD
   Planning requirements:
   → INTEGRATED multi-hazard risk assessment mandatory
   → Development must address ALL hazards
   → Consider relocation for critical infrastructure
   → Implement comprehensive DRR measures

Nairobi:
   Exposed to 3 hazard types (3 total events)
   Hazards: FIRE, FLOOD
   Planning requirements:
   → INTEGRATED multi-hazard risk assessment mandatory
   → Development must address ALL hazards
   → Consider relocation for critical infrastructure
   → Implement comprehensive DRR measures

Rift Valley:
   Exposed to 3 hazard types (2 total events)
   Hazards: EPIDEMIC, FLOOD
   Planning requirements:
   → INTEGRATED multi-hazard risk assessment mandatory
   → Development must address ALL hazards
   → Consider relocation for critical infrastructure
   → Implement comprehensive DRR measures

Western:
   Exposed to 3 hazard types (2 tota

## 7. Cost-Benefit of Risk-Informed Planning

In [8]:
if df is not None and 'usdvalue' in df.columns:
    total_losses = df['usdvalue'].sum()
    avg_annual_loss = df.groupby('year')['usdvalue'].sum().mean()
    
    print("\n" + "="*80)
    print("COST-BENEFIT OF RISK-INFORMED PLANNING")
    print("="*80)
    
    print(f"\nHistorical losses: ${total_losses/1e9:.2f} billion USD")
    print(f"Average annual loss: ${avg_annual_loss/1e6:.2f} million USD")
    
    # Estimate potential savings from risk-informed planning
    scenarios = {
        'Conservative (10% reduction)': 0.10,
        'Moderate (25% reduction)': 0.25,
        'Ambitious (40% reduction)': 0.40
    }
    
    print("\n💰 POTENTIAL SAVINGS FROM RISK-INFORMED PLANNING:\n")
    
    for scenario, reduction in scenarios.items():
        annual_savings = avg_annual_loss * reduction
        decade_savings = annual_savings * 10
        
        print(f"{scenario}:")
        print(f"   Annual savings: ${annual_savings/1e6:.2f} million USD")
        print(f"   10-year savings: ${decade_savings/1e9:.2f} billion USD")
        print(f"   Achieved through:")
        print(f"   → Restricting development in high-risk zones")
        print(f"   → Enforcing resilient building codes")
        print(f"   → Strategic infrastructure placement")
        print(f"   → Climate adaptation investments\n")
    
    print("📊 RETURN ON INVESTMENT:")
    print("   Planning cost: Minimal (integrate into existing processes)")
    print("   Benefit: Millions to billions in avoided losses")
    print("   ROI: Potentially 100:1 or higher")


COST-BENEFIT OF RISK-INFORMED PLANNING

Historical losses: $0.25 billion USD
Average annual loss: $17.96 million USD

💰 POTENTIAL SAVINGS FROM RISK-INFORMED PLANNING:

Conservative (10% reduction):
   Annual savings: $1.80 million USD
   10-year savings: $0.02 billion USD
   Achieved through:
   → Restricting development in high-risk zones
   → Enforcing resilient building codes
   → Strategic infrastructure placement
   → Climate adaptation investments

Moderate (25% reduction):
   Annual savings: $4.49 million USD
   10-year savings: $0.04 billion USD
   Achieved through:
   → Restricting development in high-risk zones
   → Enforcing resilient building codes
   → Strategic infrastructure placement
   → Climate adaptation investments

Ambitious (40% reduction):
   Annual savings: $7.18 million USD
   10-year savings: $0.07 billion USD
   Achieved through:
   → Restricting development in high-risk zones
   → Enforcing resilient building codes
   → Strategic infrastructure placement
  

## 8. Executive Summary for Planning Authorities

In [9]:
if df is not None:
    print("\n" + "="*80)
    print("EXECUTIVE BRIEFING: RISK-INFORMED DEVELOPMENT PLANNING")
    print("For: Ministry of Planning / Urban Development Authority")
    print("="*80)
    
    print("\n1. LAND USE ZONING:")
    if risk_df is not None:
        very_high = len(risk_df[risk_df['risk_score'] > 0.7])
        print(f"   {very_high} locations identified as VERY HIGH RISK")
        print("   → Prohibit new development in these zones")
        print("   → Convert to green spaces or relocate populations")
    
    print("\n2. INFRASTRUCTURE INVESTMENT:")
    print("   Historical data identifies where infrastructure repeatedly fails.")
    print("   → Prioritize resilient infrastructure in high-damage areas")
    print("   → Retrofit critical facilities (hospitals, schools)")
    
    print("\n3. DEVELOPMENT POLICY SCREENING:")
    print("   5-step screening framework for all new development proposals:")
    print("   → Location risk → Hazard exposure → Infrastructure type → Resilience → Environmental impact")
    
    print("\n4. CLIMATE ADAPTATION TARGETING:")
    climate_events = len(df[df['hazardtype'].isin(['FLOOD', 'DROUGHT', 'STORM', 'HEAVY RAIN'])])
    print(f"   {climate_events} climate-related events in historical data")
    print("   → Target adaptation investments to most affected locations")
    print("   → Implement hazard-specific adaptation strategies")
    
    print("\n5. MULTI-HAZARD CONSIDERATIONS:")
    if 'level2' in df.columns:
        multi_hazard_count = len(df.groupby('level2')['hazardtype'].nunique()[lambda x: x >= 3])
        print(f"   {multi_hazard_count} locations exposed to 3+ hazard types")
    print("   → Require integrated multi-hazard assessment")
    print("   → Development must address ALL identified hazards")
    
    print("\n6. ECONOMIC JUSTIFICATION:")
    if 'usdvalue' in df.columns:
        potential_savings = df.groupby('year')['usdvalue'].sum().mean() * 0.25
        print(f"   Potential savings (25% reduction): ${potential_savings/1e6:.2f}M annually")
    print("   → Risk-informed planning pays for itself many times over")
    print("   → Minimal cost to integrate into existing processes")
    
    print("\n7. DATA ADVANTAGE:")
    print("   ✓ Evidence-based land use zoning")
    print("   ✓ Infrastructure investment priorities")
    print("   ✓ Policy screening framework")
    print("   ✓ Climate adaptation targeting")
    print("   ✓ Multi-hazard exposure mapping")
    
    print("\n" + "="*80)
    print("RECOMMENDATION: Adopt risk-informed planning as MANDATORY for all development")
    print("="*80)


EXECUTIVE BRIEFING: RISK-INFORMED DEVELOPMENT PLANNING
For: Ministry of Planning / Urban Development Authority

1. LAND USE ZONING:
   0 locations identified as VERY HIGH RISK
   → Prohibit new development in these zones
   → Convert to green spaces or relocate populations

2. INFRASTRUCTURE INVESTMENT:
   Historical data identifies where infrastructure repeatedly fails.
   → Prioritize resilient infrastructure in high-damage areas
   → Retrofit critical facilities (hospitals, schools)

3. DEVELOPMENT POLICY SCREENING:
   5-step screening framework for all new development proposals:
   → Location risk → Hazard exposure → Infrastructure type → Resilience → Environmental impact

4. CLIMATE ADAPTATION TARGETING:
   24 climate-related events in historical data
   → Target adaptation investments to most affected locations
   → Implement hazard-specific adaptation strategies

5. MULTI-HAZARD CONSIDERATIONS:
   0 locations exposed to 3+ hazard types
   → Require integrated multi-hazard asses